# Training a Neural Network with Keras: Walkthrough + Guided Practice

**CSCI 345 · Machine Learning** · Based on Chapter 10 of *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (A. Géron)

This notebook has two parts:

| Part | What you do |
|---|---|
| **Part 1: Walkthrough** | Follow along as we train a neural network to recognise clothing in **Fashion-MNIST** images. Read each explanation, run each cell, and answer the 💭 *Think* questions in your head (or in a markdown cell). |
| **Part 2: Your turn** | Apply the *same recipe* to a new dataset, **Breast Cancer Wisconsin (Diagnostic)**, by filling in the `TODO` cells. |

### The recipe we'll follow every time

```
1. Load the data          5. Compile (loss, optimizer, metrics)
2. Explore it             6. Train (fit) and watch the learning curves
3. Pre-process it         7. Evaluate on the test set, once
4. Build the model        8. Predict, improve, save
```

> 💡 **Running in Google Colab?** *Runtime → Change runtime type → GPU* makes Part 1 faster, but a CPU is fine too (a few minutes).

---
# Part 1: Walkthrough with Fashion-MNIST

## Step 0: Set up

We use **Keras**, the high-level neural-network library that ships inside TensorFlow.
Setting a random seed makes results (nearly) repeatable, so your numbers will be close to your neighbour's.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print("TensorFlow version:", tf.__version__)
print("Keras version:     ", keras.__version__)

keras.utils.set_random_seed(42)   # seeds Python, NumPy and TensorFlow in one call

## Step 1: Load the data

**Fashion-MNIST** (from Zalando Research) has 70,000 grayscale images of clothing, each **28 × 28 pixels**, in **10 classes**.
Keras can download it for us, already split into a training set (60,000) and a test set (10,000).

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

print("X_train_full:", X_train_full.shape, X_train_full.dtype)
print("y_train_full:", y_train_full.shape, y_train_full.dtype)
print("X_test:      ", X_test.shape)

Each label is a number 0–9. This list turns a number into a name: label `2` means `class_names[2]`, which is `"Pullover"`.

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## Step 2: Explore the data

**Always look at your data before modelling it.** Three quick checks:
what do examples look like, are the classes balanced, and what range are the values in?

In [ ]:
# 1) What do the images look like?
fig, axes = plt.subplots(3, 8, figsize=(13, 5.5))
for ax, i in zip(axes.flat, range(24)):
    ax.imshow(X_train_full[i], cmap="gray")
    ax.set_title(class_names[y_train_full[i]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 2) Are the classes balanced?
counts = pd.Series(y_train_full).map(dict(enumerate(class_names))).value_counts()
print(counts)

# 3) What range are the pixel values in?
print("\nPixel values range from", X_train_full.min(), "to", X_train_full.max())

💭 **Think:** Every class has exactly 6,000 training images. Why is that good news when we judge the model by *accuracy*?

> *If one class were 90% of the data, a "model" that always guessed that class would score 90% accuracy while learning nothing.
> With balanced classes, random guessing gets only 10%.*

## Step 3: Pre-process the data

Two jobs:

1. **Scale** the pixels from 0–255 down to 0–1. Neural networks train much better when inputs are small and on a similar scale,
   because gradient descent takes sensible-size steps.
2. **Split** off a **validation set**. We now have three sets, each with one job:

| Set | Size | Used for |
|---|---|---|
| **Training** | 55,000 | Learning the weights |
| **Validation** | 5,000 | Checking progress *during* training and choosing settings (epochs, layer sizes …) |
| **Test** | 10,000 | One final, honest score **at the very end**. Never used for decisions. |

> ⚠️ If you tune your model by looking at the **test** score, the test set stops being "unseen data" and your final score becomes too optimistic.

In [ ]:
X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.0

print("train:", X_train.shape, " valid:", X_valid.shape, " test:", X_test.shape)
print("pixel range now:", X_train.min(), "to", X_train.max())

## Step 4: Build the model

We build a **multilayer perceptron (MLP)**, exactly the kind of network from our MLP lecture, with the **Sequential API**
(a simple stack of layers, one after another):

| Layer | What it does |
|---|---|
| `Input(shape=(28, 28))` | Declares the shape of one example |
| `Flatten` | Unrolls the 28 × 28 grid into a vector of **784** numbers (no learning here) |
| `Dense(300, relu)` | Hidden layer 1: 300 neurons, each a weighted sum + ReLU |
| `Dense(100, relu)` | Hidden layer 2: 100 neurons |
| `Dense(10, softmax)` | Output layer: **one neuron per class**. Softmax turns the 10 scores into probabilities that add up to 1 |

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(300, activation="relu", name="hidden_1"),
    keras.layers.Dense(100, activation="relu", name="hidden_2"),
    keras.layers.Dense(10, activation="softmax", name="output"),
])
model.summary()

### Where do the parameter counts come from?

A `Dense` layer has **(inputs × neurons) weights + (neurons) biases**:

| Layer | Calculation | Parameters |
|---|---|---|
| hidden_1 | 784 × 300 + 300 | 235,500 |
| hidden_2 | 300 × 100 + 100 | 30,100 |
| output | 100 × 10 + 10 | 1,010 |
| **Total** | | **266,610** |

That's 266,610 numbers that training must tune, and they all start out **random**.

### Sanity check before training

An untrained network should be no better than random guessing. Let's predict one image:

In [ ]:
probs = model.predict(X_valid[:1], verbose=0)[0]
print("Predicted probabilities:", np.round(probs, 3))
print("They sum to:", probs.sum().round(3))
print("Predicted:", class_names[probs.argmax()], "| Actual:", class_names[y_valid[0]])

💭 **Think:** The 10 probabilities are all roughly 0.1. Why does that make sense *before* training?

## Step 5: Compile the model

`compile()` tells Keras **how** to learn:

| Setting | Our choice | Meaning |
|---|---|---|
| `loss` | `"sparse_categorical_crossentropy"` | How wrong a prediction is. *Sparse* because our labels are integers 0–9 (with one-hot labels you'd use `"categorical_crossentropy"`). |
| `optimizer` | SGD, learning rate 0.01 | The algorithm that updates the weights: **gradient descent** using **backpropagation** |
| `metrics` | `["accuracy"]` | Extra numbers to *report*. The optimizer minimises the **loss**, not the metrics. |

**Quick rule for the output layer and loss:**

| Task | Output layer | Loss |
|---|---|---|
| Multi-class (one of *k*) | `Dense(k, "softmax")` | `sparse_categorical_crossentropy` |
| Binary (yes / no) | `Dense(1, "sigmoid")` | `binary_crossentropy` |
| Regression (a number) | `Dense(1)` (no activation) | `mse` |

*(You will use the binary row in Part 2.)*

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=0.01),
              metrics=["accuracy"])

# Loss before any training: with 10 classes, random guessing gives about -ln(1/10) = 2.30
loss0, acc0 = model.evaluate(X_valid, y_valid, verbose=0)
print(f"Before training: loss = {loss0:.2f} (expect about 2.30), accuracy = {acc0:.1%} (expect about 10%)")

## Step 6: Train the model

`fit()` runs the training loop. Key terms:

- **Batch:** Keras feeds 32 images at a time (the default). After each batch it does one forward pass, computes the loss,
  backpropagates and updates the weights. This is *mini-batch* gradient descent.
- **Epoch:** one full pass over all 55,000 training images, i.e. 55,000 / 32 ≈ **1,719 batches**.
- **`validation_data`:** after each epoch, Keras reports the loss and accuracy on the validation set. The weights are **not** trained on it.

We save what `fit()` returns in `history`, a record of every epoch.

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=30,
                    validation_data=(X_valid, y_valid))

### Read the learning curves

These plots are the most useful diagnostic you have.

In [ ]:
def plot_history(history, title=""):
    pd.DataFrame(history.history).plot(figsize=(8, 5), grid=True,
                                       xlabel="Epoch", style=["r--", "r-", "b--", "b-"])
    plt.gca().set_ylim(0, 1)
    plt.title(title)
    plt.show()

plot_history(history, "Fashion-MNIST MLP trained with SGD")

**How to read them:**

| You see | It means | Try |
|---|---|---|
| Training and validation curves stay close and are still improving | Under-trained | More epochs |
| Training keeps improving, **validation flattens or gets worse** | **Overfitting**: memorising the training set | Stop earlier, use more data, regularise (e.g. dropout) |
| Both curves are bad | Model too simple, or learning rate wrong | Bigger model, tune the learning rate |

💭 **Think:** Around which epoch does validation loss stop improving? Is there a gap between training and validation accuracy at the end?

## Step 7: Evaluate on the test set (once!)

We have finished making decisions, so now we get the honest score on data the model has never seen.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.2%}   (test loss {test_loss:.3f})")

You should see roughly **88%**. Accuracy is a single number, though. **Which** classes does the model confuse?
A **confusion matrix** shows this: rows are true classes, columns are predicted classes, and the diagonal counts the correct predictions.

In [ ]:
y_proba = model.predict(X_test, verbose=0)    # shape (10000, 10): a probability per class
y_pred = y_proba.argmax(axis=1)               # pick the most likely class

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names,
                                        xticks_rotation=45, ax=ax, colorbar=False)
plt.title("Confusion matrix (test set)")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

💭 **Think:** Which class has the lowest recall? Which classes is it most often confused with? Does that match what *you* saw in Step 2?

> *Usually **Shirt**: it gets mistaken for T-shirt/top, Pullover and Coat. Even humans struggle with these at 28 × 28.*

## Step 8: Predict, improve, save

### 8a. Look at individual predictions

The title shows *predicted (confidence)* and the true label. Green means correct, red means wrong.

In [ ]:
def show_predictions(model, X, y, n=15, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=n, replace=False)
    probs = model.predict(X[idx], verbose=0)
    fig, axes = plt.subplots(3, 5, figsize=(13, 8.5))
    for ax, i, p in zip(axes.flat, idx, probs):
        pred = p.argmax()
        ax.imshow(X[i], cmap="gray")
        ax.set_title(f"{class_names[pred]} ({p[pred]:.0%})\ntrue: {class_names[y[i]]}",
                     color="green" if pred == y[i] else "red", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(model, X_test, y_test)

### 8b. Improve: a better optimizer and early stopping

Two common upgrades:

- **Adam** optimizer: adapts the learning rate for every weight, so it usually trains faster than plain SGD.
- **Callbacks**: functions Keras calls during training.
  - `EarlyStopping` watches `val_loss` and **stops** when it hasn't improved for `patience` epochs.
    `restore_best_weights=True` then rolls back to the best epoch. So we can ask for many epochs without worrying about overfitting.
  - `ModelCheckpoint` saves the best model to disk while training.

We'll also build the model with the **Functional API** inside a function, which makes it easy to try different sizes.
In the functional style each layer is *called* on the output of the previous one:

In [ ]:
def build_model(hidden=(300, 100), dropout=0.0):
    inputs = keras.Input(shape=(28, 28))
    x = keras.layers.Flatten()(inputs)
    for units in hidden:
        x = keras.layers.Dense(units, activation="relu")(x)
        if dropout > 0:
            x = keras.layers.Dropout(dropout)(x)   # randomly switches off neurons while training
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs)

keras.utils.set_random_seed(42)
model2 = build_model(hidden=(300, 100), dropout=0.2)
model2.compile(loss="sparse_categorical_crossentropy",
               optimizer=keras.optimizers.Adam(learning_rate=1e-3),
               metrics=["accuracy"])

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint("best_fashion_mlp.keras", monitor="val_loss", save_best_only=True),
]

history2 = model2.fit(X_train, y_train, epochs=100,
                      validation_data=(X_valid, y_valid),     # validation, never test!
                      callbacks=callbacks)

plot_history(history2, "Adam + dropout + early stopping")

In [ ]:
for name, m in [("SGD, 30 epochs", model), ("Adam + dropout + early stopping", model2)]:
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{name:35s} test accuracy = {acc:.2%}")

💭 **Think:** How many epochs did early stopping actually run? Compare the gap between the training and validation curves with the first model's.

> **Optional: TensorBoard (Colab).** Add `keras.callbacks.TensorBoard("logs")` to the callbacks list, then run `%load_ext tensorboard`
> and `%tensorboard --logdir logs` to explore the curves interactively.

### 8c. Save and reload

A trained model is just a file. Save it, load it back later (or in another program), and predict, with no retraining needed.

In [ ]:
model2.save("fashion_mlp.keras")
reloaded = keras.models.load_model("fashion_mlp.keras")

same = np.allclose(model2.predict(X_test[:5], verbose=0), reloaded.predict(X_test[:5], verbose=0))
print("Reloaded model gives identical predictions:", same)

## Part 1 recap: the Keras cheat sheet

```python
model = keras.Sequential([keras.Input(shape=...),                 # 4. build
                          keras.layers.Dense(n, activation="relu"),
                          keras.layers.Dense(k, activation="softmax")])
model.compile(loss="...", optimizer="adam", metrics=["accuracy"])  # 5. compile
history = model.fit(X_train, y_train, epochs=..., validation_data=(X_valid, y_valid),
                    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])  # 6. train
model.evaluate(X_test, y_test)                                     # 7. evaluate (once)
model.predict(X_new)                                               # 8. predict
model.save("my_model.keras")                                       #    save
```

**Sequential vs Functional API:** they build the same models here. The Functional API also handles models that are *not* a single
stack: several inputs or outputs, branches, and skip connections. That is why most modern architectures use it.

---
# Part 2: Your turn. Breast cancer diagnosis

### The dataset

**Breast Cancer Wisconsin (Diagnostic)**, from the UCI Machine Learning Repository:
🔗 https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic

- **569** patients, each a fine-needle-aspirate image of a breast mass, described by **30 numeric features**
  (radius, texture, perimeter, area, smoothness … each as mean, standard error, and worst value).
- **Target:** is the mass **malignant** (cancerous) or **benign**?
- License: CC BY 4.0. It ships with scikit-learn, so no download is needed:
  [`sklearn.datasets.load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).

### What's different from Part 1?

| | Part 1 (Fashion-MNIST) | Part 2 (Breast cancer) |
|---|---|---|
| Data | Images, 784 pixels | **Tabular**, 30 measurements |
| Task | 10-class | **Binary** (malignant vs benign) |
| Size | 70,000 | **569**, small, so overfitting is a real risk |
| Scaling | Divide by 255 | Features on wildly different scales, so use **`StandardScaler`** |
| Mistakes | All equally bad | **Missing a cancer is much worse** than a false alarm |

### How to work

Fill in every cell marked **`# TODO`**, run it, and check the ✅ **Check** line before moving on.
Hints are in the comments. Answer the ✍️ questions in the markdown cells.

## P1: Load the data

`load_breast_cancer` labels **malignant = 0** and **benign = 1**. We flip that so that **1 = malignant**: the positive class should
be the one we care about *finding*, which makes *recall* mean "fraction of cancers detected".

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X_bc = data.data                 # DataFrame: 569 rows x 30 columns
y_bc = 1 - data.target           # now 1 = malignant, 0 = benign
bc_names = ["benign", "malignant"]

X_bc.head()

## P2: Explore

✍️ Answer from your output: How many malignant and benign cases are there? Is the dataset balanced?
What is the smallest and largest feature value in the whole table, and why might that hurt a neural network?

In [ ]:
# TODO 1: print the number of malignant (1) and benign (0) cases
#   hint: y_bc.value_counts()


# TODO 2: look at the feature scales
#   hint: X_bc.describe().T[["mean", "min", "max"]]  and  X_bc.values.min(), X_bc.values.max()

✅ **Check:** 212 malignant, 357 benign. Values range from 0 up to roughly 4,250.

## P3: Split and scale

1. Split into **train 60% / validation 20% / test 20%**. Use `stratify=` so every set keeps the same malignant/benign ratio.
2. **Scale** with `StandardScaler` (each feature gets mean 0 and std 1).
   **Fit the scaler on the training set only**, then `transform` all three sets.

💭 Why fit only on the training set? If the scaler sees validation or test data, information from those sets leaks into training
(**data leakage**), and your scores become too optimistic.

In [ ]:
# TODO 3: split. First take 20% off as the test set, then split the rest 75/25 into train/valid.
#   X_tmp, X_te, y_tmp, y_te = train_test_split(X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=42)
#   X_tr, X_va, y_tr, y_va  = train_test_split(...)


# TODO 4: scale. Fit on X_tr ONLY, then transform X_tr, X_va and X_te.
scaler = StandardScaler()


print("train:", X_tr.shape, " valid:", X_va.shape, " test:", X_te.shape)

✅ **Check:** train (341, 30), valid (114, 30), test (114, 30). Name your scaled arrays `X_tr_s`, `X_va_s`, `X_te_s`.

## P4: Build the model

Build a small MLP. With only 341 training examples, a 266,610-parameter network like Part 1's would overfit badly.

- Input: 30 features
- Hidden: `Dense(16, relu)`, then `Dense(8, relu)`
- Output: **binary task**, so check the table in Step 5: how many neurons, and which activation?

✍️ How many parameters does your model have? Work it out by hand first, then compare with `summary()`.

In [ ]:
keras.utils.set_random_seed(42)

# TODO 5: build the model
bc_model = keras.Sequential([
    keras.Input(shape=(30,)),
    # ... hidden layers ...
    # ... output layer: how many neurons? which activation? ...
])
bc_model.summary()

✅ **Check:** total parameters = **641**. If you got 650, your output layer has 2 neurons. Re-read the table.

## P5: Compile

Use `binary_crossentropy`, the `adam` optimizer, and track **accuracy** *and* **recall**
(`keras.metrics.Recall(name="recall")`), the fraction of malignant cases the model catches.

In [ ]:
# TODO 6: compile
bc_model.compile(
    loss=...,
    optimizer=...,
    metrics=[...],
)

## P6: Train with early stopping

Train for up to **200 epochs** with `batch_size=32`, validating on the **validation** set, and stop early when `val_loss` hasn't
improved for **15** epochs (restore the best weights). Then plot the learning curves.

In [ ]:
# TODO 7: create an EarlyStopping callback and train. Save the result in bc_history.
early = ...

bc_history = bc_model.fit(...)

# TODO 8: plot the learning curves
#   hint: pd.DataFrame(bc_history.history)[["loss", "val_loss"]].plot(grid=True, xlabel="Epoch")

✍️ After how many epochs did training stop? Does the model overfit, and how can you tell from the curves?

## P7: Evaluate on the test set

1. `evaluate` on the scaled test set.
2. Predict probabilities and turn them into labels with a **0.5 threshold**.
3. Draw the confusion matrix and print the classification report.

✍️ In a cancer screening tool, which error is worse: a **false negative** (cancer predicted as benign) or a **false positive**? How many of each did your model make?

In [ ]:
# TODO 9: evaluate on the test set
results = bc_model.evaluate(...)
print(results)

# TODO 10: probabilities -> labels (threshold 0.5)
p_te = bc_model.predict(X_te_s, verbose=0).ravel()
y_hat = ...

# TODO 11: confusion matrix + classification report (use display_labels=bc_names / target_names=bc_names)

✅ **Check:** test accuracy should be about **95% or higher**. If it is around 60%, check that you used the *scaled* arrays.

## P8: Experiment. Change ONE thing

Scientists change one variable at a time. Copy your P4–P7 code into a helper function, then try **each** experiment below, starting
again from the same seed. Record the **test accuracy** and **recall** in the table.

| # | Experiment | Test accuracy | Recall |
|---|---|---|---|
| 0 | Your baseline (P4–P7) | | |
| 1 | **No scaling**: train on `X_tr`, `X_va`, `X_te` instead of the scaled arrays | | |
| 2 | Much bigger network: `Dense(256)`, then `Dense(128)` | | |
| 3 | Add `Dropout(0.3)` after each hidden layer | | |
| 4 | SGD with `learning_rate=0.001` instead of Adam | | |

✍️ Which change hurt the most? Explain *why*, using what you know about gradient descent and feature scales.

In [ ]:
def run_experiment(hidden=(16, 8), dropout=0.0, optimizer="adam", scaled=True, seed=42):
    # TODO 12: build, compile, train (with early stopping) and evaluate a model.
    #          Return (test_accuracy, test_recall).
    #   hint: choose Xa, Xv, Xt = (X_tr_s, X_va_s, X_te_s) if scaled else (X_tr.values, X_va.values, X_te.values)
    ...

# TODO 13: run the baseline and the four experiments, and print a small table

## P9: Trade-off. Move the decision threshold

The model outputs a **probability** of malignant. Using 0.5 as the cut-off is just a default.
**Lowering** the threshold flags more cases as malignant, which catches more cancers (higher recall) but raises more false alarms.

Using your baseline `bc_model` and `p_te` from P7, compute the number of **false negatives**, **false positives** and the **recall**
for thresholds 0.5, 0.3 and 0.1.

In [ ]:
from sklearn.metrics import confusion_matrix, recall_score

for t in [0.5, 0.3, 0.1]:
    # TODO 14: labels at threshold t, then FN, FP and recall
    #   hint: tn, fp, fn, tp = confusion_matrix(y_te, labels).ravel()
    ...

✍️ Which threshold would you choose for a **screening** tool, and why? Who should make that decision: the data scientist alone, or with doctors?

## Reflection (submit with your notebook)

1. List the 8 steps of the recipe and, for each, one thing you did **differently** in Part 2 than in Part 1.
2. Why did we use `Dense(1, sigmoid)` + `binary_crossentropy` here but `Dense(10, softmax)` + `sparse_categorical_crossentropy` in Part 1?
3. What is data leakage, and where in Part 2 did we avoid it?
4. Your best model reaches, say, 97% accuracy. Would you deploy it in a hospital? What else would you want to know?

### 🚀 Stretch challenge

Try the same recipe on the **Wine Quality** dataset (UCI): https://archive.ics.uci.edu/ml/datasets/wine+quality
(direct download: https://archive.ics.uci.edu/static/public/186/wine+quality.zip, which contains `winequality-red.csv`; read it with `pd.read_csv(..., sep=";")`).
Predict a wine's quality score (0–10) as a **regression** task: last layer `Dense(1)`, loss `"mse"`, metric `"mae"`.
Then compare your mean absolute error with always predicting the mean quality.